In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

# StateBackend

In [2]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [3]:
agent = create_deep_agent(model="groq:openai/gpt-oss-20b")

agent2 = create_deep_agent(
    model="groq:openai/gpt-oss-20b",
    backend=StateBackend()
)

In [4]:
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you have done it"
        )
    }]
})

print("\n--- Agent reply ---")
print(result["messages"][-1].content)


--- Agent reply ---
✅ File `/notes/todo.txt` has been created with the requested content.


In [5]:
result

{'messages': [HumanMessage(content='Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you have done it', additional_kwargs={}, response_metadata={}, id='bbdd3c5c-e589-4a56-99be-a16d0ab3f1b6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to create file. Use write_file. Path: /notes/todo.txt. Content exactly the lines. Ensure newline at end? Provide content string. Then respond.', 'tool_calls': [{'id': 'fc_1a2c1149-bb58-4bc8-9640-b049a3f9f85b', 'function': {'arguments': '{"content":"1. Record video\\n2. Edit video\\n3. Upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 1938, 'total_tokens': 2019, 'completion_time': 0.085942095, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.097705964, 'prompt_tokens_details': None, 'queue_time': 0.016971944, 'tot

files est stoké dans le backend c'est à dire LangraphState

In [6]:
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": "Read/notes/todo.txt back to me ."
    }],
    "files": result.get("files", {})
})

print("\n--- Read-back ---")
print(result["messages"][-1].content)


--- Read-back ---
Here’s the content of **/notes/todo.txt**:

```
1. Record video
2. Edit video
3. Upload video
```



# FileSystemBackend(local disk)

In [7]:
from deepagents.backends import FilesystemBackend

ROOT = "."

agent = create_deep_agent(model="groq:openai/gpt-oss-20b",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))

print(f"root_dir={ROOT!r}")
print("Files written by the agent will appear on the actual disk")

root_dir='.'
Files written by the agent will appear on the actual disk


In [8]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you have done it"
        )
    }]
})

print("\n--- Agent reply ---")
print(result["messages"][-1].content)


--- Agent reply ---
✅ File `/notes/todo.txt` has been created with the requested content.


In [9]:
# Check backend is working
from pathlib import Path

disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print("FielsystemBackend is working. File exists on disk")
    print(f"{disk_path.resolve()}\n{'-'*40}")
    print(disk_path.read_text())

else:
    print(f"Expected file not found at {disk_path.resolve()}")
    print("The agent may not have called the write tool, or the path mapping differs")

FielsystemBackend is working. File exists on disk
C:\Users\nico_\Desktop\Agentic\deepagent\deepagentsdemo\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [10]:
fresh_agent = create_deep_agent(model="groq:openai/gpt-oss-20b",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim"
    }]
})

print(followup["messages"][-1].content)

1. Record video  
2. Edit video  
3. Upload video


# StoreBackend verification

In [11]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="groq:openai/gpt-oss-20b",
    backend=StoreBackend(
        namespace=lambda rt: ("demo-user",)
    ),
    store=store
)

print("Agent created with StoreBackend + static namespace")

Agent created with StoreBackend + static namespace


In [12]:
import os
import uuid

# write a file
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you have done it"
            )
        }]
    },
    config=thread_1
)

print(result["messages"][-1].content)

✅ File `/notes/todo.txt` has been created with the requested content.


In [13]:
# read back on a different thread

thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

followup = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Read /notes/todo.txt back to me verbatim"
        }]
    },
    config=thread_2
)

print(followup["messages"][-1].content)

1. Record video
2. Edit video
3. Upload video


La mémoire survit d'une conversation à l'autre (thread1 et thread2), sans avoir à transporter les fichiers soi-même.